In [7]:
from openai import OpenAI
import os
import json
import hashlib
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def load_json(path):
    return json.load(open(path , 'r', encoding='utf-8'))

c:\Users\ASUS\anaconda3\envs\vilegal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
day1000_comments = load_json(r'comment\day1000_comments.json')
daiphatthanh_comments = load_json(r'comment\daiphatthanh_comments.json')
theanh_comments = load_json(r'comment\theanh_comments.json')

In [15]:
len(day1000_comments['comments']) + len(daiphatthanh_comments['comments']) + len(theanh_comments['comments'])

8592

In [16]:
day1000_comments['comments']

['Đặc hay lỏng đặc thì vợ cả ăn lỏng thì cho vợ bé ăn 🐦',
 'đơ ti tót',
 'Nhìn mặt cha này hứng gì nổi mà hỏi 🙄',
 'Sao người ta nói tỉnh bơ hay ha, mình ngồi nghe cũng ngại nữa',
 'ddix mẹ:))))))))',
 'Đặc với chả lỏng tuổi đấy có khi còn kh 12h nổi ấy chứ mà',
 'sao t lại nghĩ đến m nhỉ =))))',
 'Coi lại cười khùng :)))',
 'thoại cái gì ngại vậy trời :))',
 '=)))))))))())',
 'de toi vai len bung no nha?',
 'Nó cũng hay đó chứ',
 'Đợt đấy coi ở rạp cười sặc 🤣🤣',
 'Nghệ vch í',
 'Hay nhò',
 'má ????',
 'Clm😂',
 'Cho t gia đình hp dc k ??? 😔😔😔😔',
 'gi z má',
 '=))',
 'Xin tên film cho thằng bạn',
 'Đớt ty tót',
 'đặc hay lỏng 🥴',
 'Má nó hài cốt :)))))',
 'vãi lên đâu cơ hả chú Mũi To\U0001faea',
 'Thượng phẩm',
 'thoại chấn động điên =)))',
 'Coi mắc cõ lun á má',
 'gì mắc cỡ z trời',
 'Ghé trang mình có đồ xinh 🛍️',
 'thoại chấn động=))',
 'Elite ball knowledge',
 'quả cap bà ui',
 'vlxx =))',
 'như lước vo gạo :))',
 'Phim j z ạ',
 '=)))',
 '"tôi vãi lên bụng nó nhá"',
 ':)))))))))))

In [3]:
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

In [4]:
messages = [{"role": "user", 
             "content": "Who are you"}]

completion = client.chat.completions.create(
    model="qwen3-max",  # You can replace this with another deep thinking models
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)

In [5]:
is_answering = False  # Indicates whether the response phase has started
print("\n" + "=" * 20 + "Thinking process" + "=" * 20)
for chunk in completion:
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "Full response" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)


====================Thinking process====================
We are asked: "Who are you?" This is a straightforward question about my identity.
As an AI language model, I should introduce myself clearly and concisely.
I should mention:
- My name: Qwen
- That I am a large language model developed by Alibaba Cloud
- My capabilities: answering questions, creating text, etc.
- My purpose: to assist users
Additionally, I should be polite and redirect the user to ask for help if needed.

However, note that the user might be testing if I know my own identity. I should avoid any confusion with other models.

Let me draft:
"Hello! I am Qwen, a large language model developed by Alibaba Cloud. I am designed to assist users by providing information, answering questions, and helping with various tasks such as writing, coding, and more. How can I assist you today?"

But wait, the question is "Who are you?" and I should be precise. Also, note that the model is specifically from the Alibaba Group, and Al

In [ ]:
import re

def build_vietnamese_normalization_prompt(comments):
    """
    Build a prompt to normalize Vietnamese comments while preserving emoji/emoticons.

    Args:
        comments: str or list[str]

    Returns:
        str: prompt text ready to send to the model
    """
    if isinstance(comments, str):
        comments = [comments]

    prompt = f"""
Bạn là bộ chuẩn hóa văn bản tiếng Việt.
Nhiệm vụ: chuẩn hóa chính tả, dấu, viết tắt, teencode về tiếng Việt chuẩn.

QUY TẮC BẮT BUỘC:
1) Giữ nguyên emoji (ví dụ: 😂🔥❤️) và emoticon/ký tự cảm xúc (ví dụ: :))), :v, ^^, T_T, <3).
2) Không dịch sang ngôn ngữ khác.
3) Không thêm hoặc bớt ý nghĩa nội dung.
4) Trả về đúng định dạng JSON array, cùng số phần tử như input.
5) Mỗi phần tử output là câu đã chuẩn hóa tương ứng với câu input cùng vị trí.

Input comments (JSON array):
{json.dumps(comments, ensure_ascii=False, indent=2)}

Chỉ trả về JSON array, không thêm markdown/code block.
""".strip()
    return prompt


def _extract_json_array(text):
    """Extract the first JSON array from model output."""
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n|\n```$", "", text.strip())

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except Exception:
        pass

    match = re.search(r"\[[\s\S]*\]", text)
    if not match:
        raise ValueError("Model output does not contain a JSON array.")

    data = json.loads(match.group(0))
    if not isinstance(data, list):
        raise ValueError("Parsed JSON is not an array.")
    return data


def normalize_vietnamese_comments(client, comments, model="qwen3-max"):
    """
    Normalize Vietnamese comments using LLM.

    Returns:
        list[str]
    """
    single_input = isinstance(comments, str)
    comments_list = [comments] if single_input else list(comments)

    prompt = build_vietnamese_normalization_prompt(comments_list)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Bạn là trợ lý chuẩn hóa văn bản tiếng Việt. Tuân thủ output JSON chính xác."
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    raw_text = response.choices[0].message.content or ""
    normalized = _extract_json_array(raw_text)

    if len(normalized) != len(comments_list):
        raise ValueError(
            f"Output length mismatch: expected {len(comments_list)}, got {len(normalized)}"
        )

    if single_input:
        return normalized[0]
    return normalized


# Example:
# comments = ["tr oi dep qa :)))", "nay zui vler 😂😂"]
# normalized = normalize_vietnamese_comments(client, comments)
# print(normalized)